<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Updated-Prompt/mnps_eval_reliability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Prompt Notebook — Ground Truth–Guided Evaluation (Colab Ready)

**Purpose.** Use a **Ground Truth Masterfile** as few‑shot exemplars to guide evaluation of a **Sample File**, then save **all run artifacts** to Google Drive at:
`/content/drive/My Drive/Colab Notebooks/Run Results`

**What you need to provide in Colab:**
1) Upload your CSVs to `/content/`:
   - `Ground Truth Masterfile.csv`
   - `Sample File.csv`
2) Set your `OPENAI_API_KEY` in the Colab session environment (e.g., `os.environ['OPENAI_API_KEY'] = 'sk-...'`).

Run cells top‑to‑bottom. 

In [ ]:
# %% [markdown]
# 0) Setup — installs (Colab)

In [ ]:
# If needed, uncomment to install packages in Colab
# !pip install -q openai>=1.40 scikit-learn pydantic


In [ ]:
# 1) Imports & core config
import os, json, time, textwrap
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- FILE PATHS ---
GROUND_TRUTH_CSV = "/content/Ground Truth Masterfile.csv"
SAMPLE_CSV       = "/content/Sample File.csv"

# Directory for local run artifacts
RUN_LOCAL_DIR = "/content/run_artifacts"
Path(RUN_LOCAL_DIR).mkdir(parents=True, exist_ok=True)

# Optional: pin a run id/timestamp for traceability
RUN_ID = time.strftime("%Y%m%d_%H%M%S")

print({
    'GROUND_TRUTH_CSV': GROUND_TRUTH_CSV,
    'SAMPLE_CSV': SAMPLE_CSV,
    'RUN_LOCAL_DIR': RUN_LOCAL_DIR,
    'RUN_ID': RUN_ID
})

In [ ]:
# 2) Text/label columns & helpers
TEXT_COLS = [
    "Position Summary",
    "Essential Functions",
    "Education",
    "Work Experience",
    "Licenses and Certifications",
    "Knowledge, Skills and Abilities",
]

LABEL_COLS = [
    "New Job Title", "Major Role Group", "Minor Sub-Group",
    "Major Role", "Minor Role",  # fallback names if your GT uses these
    "MNPS Title"                   # another common fallback
]

def coalesce_cols(df, cols):
    existing = [c for c in cols if c in df.columns]
    return existing


def make_text_blob(row, cols):
    parts = []
    for c in cols:
        val = str(row.get(c, "") or "").strip()
        if val and val.lower() not in {"nan", "none"}:
            parts.append(f"{c}: {val}")
    return "\n".join(parts)

# Load data
print("Loading CSVs…")
gt = pd.read_csv(GROUND_TRUTH_CSV)
sample = pd.read_csv(SAMPLE_CSV)

text_cols_gt      = coalesce_cols(gt, TEXT_COLS)
text_cols_sample  = coalesce_cols(sample, TEXT_COLS)
label_cols_gt     = coalesce_cols(gt, LABEL_COLS)

# Build blobs
gt["__blob__"] = gt.apply(lambda r: make_text_blob(r, text_cols_gt), axis=1)
sample["__blob__"] = sample.apply(lambda r: make_text_blob(r, text_cols_sample), axis=1)

# TF-IDF fit on Ground Truth universe
vec = TfidfVectorizer(min_df=2, ngram_range=(1,2), stop_words="english")
gt_matrix = vec.fit_transform(gt["__blob__"].fillna(""))


def get_topk_exemplars(query_text, k=5):
    if not query_text.strip():
        return gt.iloc[:0]  # empty
    q = vec.transform([query_text])
    sims = cosine_similarity(q, gt_matrix)[0]
    topk_idx = np.argsort(-sims)[:k]
    out = gt.iloc[topk_idx].copy()
    out["__sim__"] = sims[topk_idx]
    return out


def format_exemplars_for_prompt(df_ex):
    # Keep things compact to avoid token bloat
    exemplars = []
    for _, r in df_ex.iterrows():
        # Extract the best label columns we have
        label_bits = []
        for c in label_cols_gt:
            if c in r and pd.notna(r[c]) and str(r[c]).strip():
                label_bits.append(f"{c}: {r[c]}")
        label_text = "; ".join(label_bits) if label_bits else "Labels: (not available)"

        # A short blob (truncate to ~1200 chars to keep prompts small)
        blob = r["__blob__"][:1200]
        exemplars.append(
            f"- EXAMPLE\n{label_text}\nTEXT\n{blob}\n"
        )
    return "\n".join(exemplars) if len(exemplars) else "None"

print("Ground Truth rows:", len(gt))
print("Sample rows:", len(sample))

In [ ]:
# 3) Schema (Pydantic) for structured output
from typing import List, Optional
from pydantic import BaseModel

class JobClassification(BaseModel):
    new_job_title: str
    major_role_group: str
    minor_sub_group: Optional[str] = None
    grouping_justification: str

class JobClassificationTable(BaseModel):
    job_classification_table: List[JobClassification]

print("Schema ready.")

In [ ]:
# 4) System prompt that uses Ground Truth exemplars in the user message
CLASSIFIER_SYSTEM_PROMPT = """\
You are an MNPS job classification assistant. Classify a single job description into:
- New Job Title (format: "[Function] [Role] [Level]" e.g., "Collections Specialist II")
- Major Role Group (one of: Specialist, Analyst, Director, Manager, Technician, Coordinator)
- Minor Sub-Group (I, II, III, IV; use the closest level; omit 'IV' if inappropriate)
- Grouping Justification (brief, cites signals across Position Summary, Essential Functions, Education, Experience, Licenses/Certifications, KSAs)

CRITICAL RULES:
- Focus on **what the job does** (functions, scope, decision latitude, supervision, consequences of error).
- **Do NOT** overweight literal job-title strings that appear in Position Summary/Essential Functions; treat them as weak hints.
- Heavily weight licensure requirements and scope of responsibility when present.
- Prefer consistency with verified MNPS decisions shown in the EXEMPLARS block.
- If unsure between adjacent levels (e.g., II vs III), choose the more conservative (lower) level and explain why.

Return structured output per the provided schema.
"""

print("Classifier system prompt set.")

In [ ]:
# 5) OpenAI client
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

if not os.environ.get("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY not set in environment; set it before running the batch.")
else:
    print("OpenAI API key detected.")

In [ ]:
# 6) Classify function + batch run over Sample File

def classify_one_record(row, k_ex=5):
    # Build exemplar block from GT
    ex_df = get_topk_exemplars(row["__blob__"], k=k_ex)
    exemplars_block = format_exemplars_for_prompt(ex_df)

    # Compose user payload from the Sample row
    job_title_original = str(row.get("Job Description Name", "") or "").strip()
    user_blob = f"""\
SAMPLE (to classify)
Original Job Title: {job_title_original or "(unknown)"}

TEXT
{row["__blob__"]}

EXEMPLARS (verified MNPS decisions from Ground Truth; use for guidance)
{exemplars_block}
"""

    messages = [
        {"role": "developer", "content": CLASSIFIER_SYSTEM_PROMPT},
        {"role": "user", "content": user_blob}
    ]

    resp = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=messages,
        temperature=0.2,
        max_tokens=900,
        response_format=JobClassificationTable
    )

    parsed = resp.choices[0].message.parsed
    jc = parsed.job_classification_table[0]
    out = {
        "run_id": RUN_ID,
        "prompt_version": "gt_exemplars_v1",
        "job_title_original": job_title_original,
        "new_job_title": jc.new_job_title,
        "major_role_group": jc.major_role_group,
        "minor_sub_group": jc.minor_sub_group,
        "grouping_justification": jc.grouping_justification,
        "n_exemplars": len(ex_df),
        "exemplar_indices": ",".join(map(str, ex_df.index.tolist())),
    }
    return out

# ---- Run over SAMPLE ----
results = []
for i, row in sample.iterrows():
    try:
        results.append(classify_one_record(row, k_ex=5))
    except Exception as e:
        results.append({
            "run_id": RUN_ID,
            "prompt_version": "gt_exemplars_v1",
            "error": str(e),
            "row_index": i
        })

df_results = pd.DataFrame(results)
out_csv_path = f"{RUN_LOCAL_DIR}/model_outputs_{RUN_ID}.csv"
df_results.to_csv(out_csv_path, index=False)
print(f"Wrote: {out_csv_path}")

df_results.head()

In [ ]:
# 7) Optional — Adjudication sheet scaffold
adj_cols = [
    "run_id","job_title_original","new_job_title",
    "major_role_group","minor_sub_group","grouping_justification",
    "review_correct_title (Y/N)","review_correct_major (Y/N)","review_correct_minor (Y/N)",
    "review_notes"
]

adj = df_results.reindex(columns=[c for c in adj_cols if c in df_results.columns] +
                                   [c for c in adj_cols if c not in df_results.columns])
adj_path = f"{RUN_LOCAL_DIR}/MNPS_Adjudication_Sheet_{RUN_ID}.csv"
adj.to_csv(adj_path, index=False)
print(f"Wrote: {adj_path}")
adj.head(3)

In [ ]:
# 8) Export all run artifacts to Google Drive
from google.colab import drive
from pathlib import Path
import glob, shutil

drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/My Drive/Colab Notebooks/Run Results"
run_dir_drive = f"{DRIVE_ROOT}/RUN_{RUN_ID}"

# Ensure destination exists
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(run_dir_drive).mkdir(parents=True, exist_ok=True)

# Also copy any CSV/JSON we created in /content (just in case)
extra_globs = [
    "/content/*.csv",
    "/content/*.json",
    "/content/*.jsonl",
]

# Copy our organized artifacts
for p in Path(RUN_LOCAL_DIR).glob("*"):
    try:
        shutil.copy2(str(p), run_dir_drive)
    except Exception as e:
        print(f"Skip {p}: {e}")

# Copy extras (top-level outputs you may have)
for g in extra_globs:
    for p in glob.glob(g):
        try:
            shutil.copy2(p, run_dir_drive)
        except Exception as e:
            print(f"Skip {p}: {e}")

print(f"Copied run artifacts to: {run_dir_drive}")

---

### Notes
- If your Ground Truth labels use different column names, add them to `LABEL_COLS`.
- You can change exemplar count via `k_ex` in `classify_one_record`.
- If you want to **ignore** title strings even more aggressively, you can strip any literal job-title patterns from the Sample blob before vectorizing.
